In [1]:
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse, StreamingResponse
from pydantic import BaseModel
from dotenv import load_dotenv
import os, uuid, json, asyncio
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage, HumanMessage
from langchain_community.utilities import OpenWeatherMapAPIWrapper
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict
from typing import Annotated
env_path = "/Users/ahmedibrahim/Desktop/Mids/projects_2026/.env"
load_dotenv(env_path)

True

In [2]:

openai_api_key = os.getenv("OPENAI_API_KEY")
open_weather_api_key = os.getenv("OPENWEATHER_API_KEY")
weather = OpenWeatherMapAPIWrapper(
    openweathermap_api_key=os.getenv("OPENWEATHER_API_KEY")
)

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.5, openai_api_key=openai_api_key)
# weather = OpenWeatherMapAPIWrapper()

In [3]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    city: str


In [4]:
def agent(state: State):
    user_input = state["messages"][-1].content
    res = llm.invoke([
        HumanMessage(content=f"""
You are given a question and must extract the city name from it.
Respond ONLY with the city name. If no city is found, respond with an empty string.
Question: {user_input}
""")
    ])
    city_name = res.content.strip()
    if not city_name:
        return {"messages": [AIMessage(content="I couldn't find a city name in your question.")]}
    return {"messages": [AIMessage(content=f"Extracted city: {city_name}")], "city": city_name}

In [5]:
def weather_tool(state: State):
    city_name = state.get("city", "").strip()
    if not city_name:
        return {"messages": [AIMessage(content="No city name provided. Cannot fetch weather.")]}
    weather_info = weather.run(city_name)
    return {"messages": [AIMessage(content=weather_info)]}

In [6]:
memory = MemorySaver()
workflow = StateGraph(State)
workflow.add_node("agent", agent)
workflow.add_node("weather", weather_tool)

workflow.add_edge(START, "agent")
workflow.add_edge("agent", "weather")
workflow.add_edge("weather", END)
graph_app = workflow.compile(checkpointer=memory)

In [7]:
class AskRequest(BaseModel):
    question: str
    stream: bool = False

app = FastAPI()
@app.post("/ask")
async def ask(req: AskRequest):
    question = req.question
    stream = req.stream
    if not question:
        return JSONResponse(content={"error": "No question provided"}, status_code=400)
    config = {"configurable": {"thread_id": str(uuid.uuid4())}}
    result = graph_app.invoke({"messages": [HumanMessage(content=question)]}, config=config)
    final_response = result["messages"][-1].content
    if stream:
        async def event_stream():
            for chunk in final_response:
                yield f"data: {json.dumps({'choices': [{'delta': {'content': chunk}}]})}\n\n"
                await asyncio.sleep(0.15)
            yield "data: [DONE]\n\n"
        return StreamingResponse(event_stream(), media_type="text/event-stream")
    else:
        return JSONResponse(content={"response": final_response})


In [8]:
import uuid
from langchain_core.messages import HumanMessage

question = "what is the weather in charlotte?"

config = {"configurable": {"thread_id": str(uuid.uuid4())}}

result = graph_app.invoke(
    {"messages": [HumanMessage(content=question)]},
    config=config
)

final_response = result["messages"][-1].content
print(final_response)

In Charlotte, the current weather is as follows:
Detailed status: few clouds
Wind speed: 4.12 m/s, direction: 320°
Humidity: 55%
Temperature: 
  - Current: -1.63°C
  - High: -0.81°C
  - Low: -2.38°C
  - Feels like: -6.37°C
Rain: {}
Heat index: None
Cloud cover: 20%


In [9]:
import uuid, asyncio
from langchain_core.messages import HumanMessage

async def ask_notebook_stream(question: str, delay: float = 0.05):
    config = {"configurable": {"thread_id": str(uuid.uuid4())}}
    result = graph_app.invoke({"messages": [HumanMessage(content=question)]}, config=config)
    final_response = result["messages"][-1].content

    # stream character-by-character (like your FastAPI for loop)
    for ch in final_response:
        print(ch, end="", flush=True)
        await asyncio.sleep(delay)

    print()  # newline at end
    return final_response

# Run it (Jupyter supports top-level await in most setups)
await ask_notebook_stream("what is the weather in new york?", delay=0.02)

In New York, the current weather is as follows:
Detailed status: snow
Wind speed: 15.95 m/s, direction: 350°
Humidity: 95%
Temperature: 
  - Current: -1.26°C
  - High: -0.44°C
  - Low: -2.25°C
  - Feels like: -8.26°C
Rain: {}
Heat index: None
Cloud cover: 100%


'In New York, the current weather is as follows:\nDetailed status: snow\nWind speed: 15.95 m/s, direction: 350°\nHumidity: 95%\nTemperature: \n  - Current: -1.26°C\n  - High: -0.44°C\n  - Low: -2.25°C\n  - Feels like: -8.26°C\nRain: {}\nHeat index: None\nCloud cover: 100%'

In [42]:
asyncio.run(ask_notebook_stream("what is the weather in new york?"))

In New York, the current weather is as follows:
Detailed status: snow
Wind speed: 15.43 m/s, direction: 360°
Humidity: 94%
Temperature: 
  - Current: -1.08°C
  - High: -0.31°C
  - Low: -1.9°C
  - Feels like: -8.08°C
Rain: {}
Heat index: None
Cloud cover: 100%


'In New York, the current weather is as follows:\nDetailed status: snow\nWind speed: 15.43 m/s, direction: 360°\nHumidity: 94%\nTemperature: \n  - Current: -1.08°C\n  - High: -0.31°C\n  - Low: -1.9°C\n  - Feels like: -8.08°C\nRain: {}\nHeat index: None\nCloud cover: 100%'

In [ ]:
# !uvicorn main:app --reload

INFO:     Will watch for changes in these directories: ['/Users/ahmedibrahim/Desktop/Mids/projects_2026/langgraph_openai/LangGraphForBeginners']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [71557] using StatReload
ERROR:    Error loading ASGI app. Could not import module "main".
^C


In [ ]:
# import nest_asyncio
# nest_asyncio.apply()

# import uvicorn
# uvicorn.run(app, port=8000)

RuntimeError: Runner.run() cannot be called from a running event loop